# design/26 — spike G_3: the review round trip

**Run this on the rig, in a browser, with your hand on the mouse.** It is the half of
`design/26-ilastik-imjoy-spike.py` that a script cannot do. Run that script first — its
G_1/G_2 stages are the diagnosis if anything below fails to open.

## What this measures, and why it is a notebook

design/26 adopts ImJoy + Kaibu as the review surface for two findings that both write
cheques against a UI nobody has built:

* **F6** — a lab with no API budget puts a *human* in the adjudicator seat, and the doc
  admits its first draft waved at this: "true of the seam and false of the experience."
  Verdicts are **training labels**, so a careless one is worse than none.
* **F3** — the operating point must be chosen *by looking*, and spike D's UMAP map is
  "the artifact to hand the adjudicator." Hand it to them **where?**

And it settles the contract `train_roi_detector` currently states as fact:

> `example_boxes` — optional per-image `(x, y, w, h)` ... Boxes are plain `(x, y, w, h)`
> in image pixels.

**Nobody has seen a box come back.** If it arrives as normalised floats, as polygon
vertices, in display coordinates against a scaled canvas, or with y flipped, that contract
is written against a fiction — and the doc dodged MM's rectangle tool precisely to avoid an
unverified coordinate question, so landing on a *second* one would be funny rather than fine.

It is a notebook because ImJoy's `api` is injected by a plugin runtime — a Jupyter kernel
with `imjoy-jupyter-extension`, or the ImJoy web app. **microclaw is a CLI.** If the window
only ever opens here, then `review_roi_candidates(surface="imjoy")` is not a rendering
choice, it is *a requirement that the operator be in a notebook*, and that belongs in
design/26 as a constraint on the F6 fallback rather than a parameter default. **That result
is worth as much as a working window.**

## The rule this spike must not break

The bar is *no network call at image time*. Everything here runs with **the stage parked and
the camera idle** — F7's one model-shaped step. Nothing below may end up imported by
`ROIDetectorHook`, `image_analysis`, or an exported hook.

## Record what you see

Every cell prints what it *got*, not what we expected. Paste the output into design/26
(F5/F6) — including the failures. A cell that raises is a measurement.

## Cell 1 — the environment

Install (on the rig, in microclaw's env — whether this is *clean alongside pycro-manager*
is itself the first measurement):

```
pip install imjoy imjoy-jupyter-extension
jupyter notebook design/26-imjoy-review-spike.ipynb
```

In [ ]:
import platform, sys

for pkg in ("imjoy", "imjoy_rpc", "hypha_rpc", "pycromanager", "numpy", "skimage"):
    try:
        mod = __import__(pkg)
        print(f"{pkg:16s} {getattr(mod, '__version__', 'imported')}")
    except Exception as exc:
        print(f"{pkg:16s} NOT IMPORTABLE — {type(exc).__name__}: {exc}")

print(f"\npython {platform.python_version()} on {platform.system()} {platform.release()}")
print(f"kernel: {'yes' if 'ipykernel' in sys.modules else 'NO — api will not be injected'}")

## Cell 2 — the image

Prefers a **real frame off the rig camera**, through microclaw's own `snap_to_numpy`, so
what crosses the bridge is what an acquisition would hand it: real dtype, real bit depth,
real geometry. Falls back to a tile the `.py` spike wrote, then to synthetic.

Note the shape and dtype it prints. If a box comes back in the wrong coordinate space, this
is the array you compare against — and a non-square frame is what exposes an (x, y) vs
(row, col) swap, so **prefer the real camera**: a square synthetic tile hides it.

In [ ]:
from pathlib import Path
import numpy as np

TILES_DIR = Path("d26_tiles/score")   # whatever you passed the .py spike as --tiles-dir
image = None

try:
    from microclaw.controller import MicroscopeController
    from microclaw.image_analysis import snap_to_numpy
    image = snap_to_numpy(MicroscopeController(port=4827))
    source = "the rig camera, via microclaw.snap_to_numpy"
except Exception as exc:
    print(f"no camera ({type(exc).__name__}: {exc}); falling back to a file")
    tifs = sorted(TILES_DIR.glob("score_*.tif"))
    if tifs:
        from skimage.io import imread
        image, source = imread(str(tifs[0])), f"{tifs[0]}"
    else:
        rng = np.random.default_rng(0)
        image = rng.normal(400, 15, (256, 320)).astype(np.uint16)  # deliberately NON-square
        source = "synthetic (non-square, so an axis swap cannot hide)"

print(f"source : {source}")
print(f"shape  : {image.shape}   (rows, cols)")
print(f"dtype  : {image.dtype}   range [{image.min()}, {image.max()}]")

## Cell 3 — does a window open at all?

The pattern is pycro-manager's own ImJoy tutorial, which design/26 read at source:
`from imjoy import api`, a plugin class, `api.export(...)`, and `api.createWindow(src=...)`.

**Two things to watch, and neither is the picture:**

1. `src="https://kaibu.org/#/app"` **fetches executable UI over the network** — which is
   exactly what design/26 forbids at F4-one-layer-up ("microclaw ships and self-hosts the
   review plugin it uses; it does not `createWindow` a gist at runtime"). It is used *here*
   because this spike is measuring whether the bridge works at all, and self-hosting a
   plugin we have not proven we want is the wrong order. **If it does not load, that is
   G_2's egress answer arriving the hard way**, and it means self-hosting is not a
   preference — it is the only implementation that runs on an isolated rig.
2. What `createWindow` **returns**. Everything downstream is method calls on that object,
   and the next cell lists what it actually exposes rather than trusting these names.

In [ ]:
from imjoy import api

viewer = None

class ReviewSpike:
    async def setup(self):
        pass

    async def run(self, ctx):
        global viewer
        viewer = await api.createWindow(src="https://kaibu.org/#/app", name="design/26 review")
        print(f"createWindow returned: {type(viewer)}")
        await viewer.add_image(image, name="tile")
        print("add_image: returned without raising")

api.export(ReviewSpike())

*(Run the plugin from the ImJoy toolbar / the cell's output widget if it does not start on
its own — the extension decides that, not us. If `viewer` is still `None` below, say so in
the write-up: a surface that needs a manual click per review is a finding about F6's
"human in the seat" ergonomics.)*

## Cell 4 — what does the remote object actually expose?

**This cell is the point of the notebook.** design/26 names `add_image`, `add_shapes`,
`add_points` from Kaibu's docs. This asks the object in front of us, on the version the rig
installed. Anything the doc names that is missing here is a design correction, not a typo.

In [ ]:
if viewer is None:
    print("viewer is None — cell 3 did not complete. That IS the G_3 result: write it down.")
else:
    methods = sorted(m for m in dir(viewer) if not m.startswith("_"))
    print(f"{len(methods)} members on the remote viewer proxy:\n")
    for m in methods:
        print(f"  {m}")
    print("\ndesign/26 names these — present?")
    for m in ("add_image", "add_shapes", "add_points", "get_layer", "clear_layers"):
        print(f"  {m:16s} {'yes' if m in methods else 'NOT PRESENT — the doc is guessing'}")

## Cell 5 — draw two rectangles, and see what comes back

**The measurement `train_roi_detector(example_boxes=...)` is waiting on.**

Run the cell, then in the Kaibu window: select the shapes layer, draw **two rectangles** —
one in the **top-left** corner, one in the **bottom-right**. Make them obviously different
sizes. Then run cell 6.

Adapt the call if cell 4 says these arguments do not exist; the exact spelling is the rig's
to tell us. Corner placement is deliberate: it is what makes an axis swap or a flipped y
**visible** rather than plausible.

In [ ]:
async def add_shapes_layer():
    layer = await viewer.add_shapes([], name="boxes", shape_type="rectangle",
                                    edge_color="#ff0000")
    print(f"add_shapes returned: {type(layer)}")
    print(f"members: {sorted(m for m in dir(layer) if not m.startswith('_'))}")
    return layer

# In a kernel with the extension loaded, `await` works at top level.
layer = await add_shapes_layer()

## Cell 6 — the coordinates

Read the boxes back **after** you have drawn them. Compare against cell 2's `shape`:

| what comes back | what it means for design/26 |
|---|---|
| ints/floats spanning `0..cols`, `0..rows` | `example_boxes` as **image pixels** is right — the contract stands |
| floats in `0..1` | normalised: the tool must multiply, and by **which** axis order? |
| 4+ vertex pairs | it is a **polygon**, not `(x, y, w, h)` — the contract changes shape |
| top-left box reports large y | **y is flipped**, and a silent flip is a mislabelled example |
| x/y swapped vs the corners you drew | (row, col) vs (x, y) — the classic, and why cell 2 prefers non-square |

Whatever it is, **paste the raw repr into design/26**. Do not summarise it into the shape
the doc already claims — design/20 and design/21 are both about exactly that move.

In [ ]:
shapes = await viewer.get_layer("boxes")     # if cell 4 says this is not the name, use its
print(f"type   : {type(shapes)}")            # list; the point is to print the REPR, not to
print(f"repr   :\n{shapes!r}")               # be right first time

print(f"\nthe image was {image.shape} (rows, cols) = ({image.shape[0]} tall, {image.shape[1]} wide)")
print("do the numbers above span those ranges, in that order, with the top-left box small?")

## Cell 7 — labels in: the seam, end to end

F6's contract is **crops out, labels in**, and `review_roi_candidates(surface="imjoy")`
promises `refine_roi_detector` "cannot tell which one produced the verdicts, beyond the
`adjudicator` string it is told." This is the cheapest possible test of that: a grid of
crops, a keep/discard per crop, and the verdicts landing back in Python as a dict.

If the callback never fires, the surface cannot carry F6's human — and `surface="blocks"`
(image blocks to an agent) remains the only shipped path.

In [ ]:
verdicts = {}

async def keep(crop_id):
    verdicts[crop_id] = True
    print(f"kept {crop_id}   -> verdicts now {verdicts}")

async def discard(crop_id):
    verdicts[crop_id] = False
    print(f"discarded {crop_id} -> verdicts now {verdicts}")

# Does a python callback survive the bridge and land back in this kernel? That is the
# whole seam. The button spelling is Kaibu's; cell 4's member list is the ground truth.
try:
    await viewer.set_ui({"title": "adjudicate",
                         "elements": [
                             {"type": "button", "label": "keep crop_0",
                              "callback": lambda: keep("crop_0")},
                             {"type": "button", "label": "discard crop_0",
                              "callback": lambda: discard("crop_0")},
                         ]})
    print("set_ui returned; click a button in the window, then re-run this cell's last line")
except Exception as exc:
    print(f"set_ui -> {type(exc).__name__}: {exc}")
    print("If this API does not exist, cell 4's member list is the answer to what does.")

---

## Write-up — what to carry back into design/26

Answer these in the doc, in the finding they belong to. **A no is as valuable as a yes**;
the point of this spike is to stop F6 and F3 resting on a UI nobody opened.

1. **Did a window open on the rig at all?** (F6, and G_2 if the CDN is unreachable.)
2. **Did it need Jupyter?** If yes → `surface="imjoy"` is a constraint on the F6 fallback,
   not a parameter default, and design/26 should say so plainly: the human adjudicator
   needs a notebook next to the CLI.
3. **What coordinate space do boxes come back in?** Paste the raw repr. This decides
   `train_roi_detector(example_boxes=...)`'s contract, which currently asserts
   "plain `(x, y, w, h)` in image pixels" on no evidence.
4. **Did a Python callback fire from a browser click?** That is F6's seam — crops out,
   labels in — and everything the doc says about a human adjudicator depends on it.
5. **Install cost:** did `pip install imjoy imjoy-jupyter-extension` disturb pycro-manager
   or pyjavaz on the rig's Python? The `review` extra is only cheap if it is.

And the standing one, from the doc's own list of things it does not do: *"nobody has watched
a human use one."* You are the first. Whether a crop grid plus a UMAP plus a **k** slider
actually produces better verdicts than scrolling crops past a chat transcript is a design,
not a measurement — and F6 is the finding that says a careless verdict is a bad **training
label**, not just a bad call.